# Loading CSV into DataFrame

- In the previous exercise, you have seen a method of creating DataFrame but generally, loading data from CSV file is the most common method of creating DataFrames. In this exercise, you'll create a PySpark DataFrame from a `people.csv` file that is already provided to you as a `file_path` and confirm the created object is a PySpark DataFrame.

- Remember, you already have `SparkSession` `spark` and `file_path` variable (which is the path to the `people.csv` file) available in your workspace..

## Instructions

- Create a DataFrame from `file_path` variable which is the path to the `people.csv` file.
- Confirm the output as PySpark DataFrame.

In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('yarn'). \
#     getOrCreate()

In [5]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [6]:
sc = spark.sparkContext

In [7]:
import pyspark.sql.functions as F


file_path = "file:///home/talentum/Lab_practice/mini/Dataset/london_house_price_data.csv"
DF = spark.read.csv(file_path , header=True , inferSchema= True)

DF.printSchema()

root
 |-- fullAddress: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- outcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- floorAreaSqM: double (nullable = true)
 |-- livingRooms: double (nullable = true)
 |-- tenure: string (nullable = true)
 |-- propertyType: string (nullable = true)
 |-- currentEnergyRating: string (nullable = true)
 |-- rentEstimate_lowerPrice: double (nullable = true)
 |-- rentEstimate_currentPrice: double (nullable = true)
 |-- rentEstimate_upperPrice: double (nullable = true)
 |-- saleEstimate_lowerPrice: double (nullable = true)
 |-- saleEstimate_currentPrice: double (nullable = true)
 |-- saleEstimate_upperPrice: double (nullable = true)
 |-- saleEstimate_confidenceLevel: string (nullable = true)
 |-- saleEstimate_ingestedAt: timestamp (nullable = true)


In [8]:
DF.select("postcode", "saleEstimate_currentPrice").printSchema()

root
 |-- postcode: string (nullable = true)
 |-- saleEstimate_currentPrice: double (nullable = true)



In [9]:
#1. Find the postcode with the highest average property sale price (saleEstimate_currentPrice). 


avgsales = DF.groupby('postcode').agg(F.avg('saleEstimate_currentPrice').alias('avgSales') )
avgsales.show(10)

MaxavgPostCode = avgsales.orderBy('avgSales', ascending=False).limit(1)
    
MaxavgPostCode.show()


#Round Up val

# MaxavgPostCode.select(
#     'postcode',
#     F.format_number('avgSales', 2).alias('avgSales')
# ).show()


+--------+-----------------+
|postcode|         avgSales|
+--------+-----------------+
|  E1 6QR|        3602000.0|
|  E1 6EB|         442500.0|
|  E1 2FF|         420000.0|
| E10 6LJ|         739750.0|
| E11 4YJ|541333.3333333334|
| E12 5HW|343833.3333333333|
| E12 5SB|714428.5714285715|
| E12 5QG|         523500.0|
| E13 0LU|         594000.0|
| E13 8DE|421571.4285714286|
+--------+-----------------+
only showing top 10 rows

+--------+--------+
|postcode|avgSales|
+--------+--------+
|SW1X 8HB|2.5681E7|
+--------+--------+



In [10]:
#2. Find the property type (propertyType) with the highest average number of bathrooms. 

#DF.select('propertyType', 'bathrooms').show()

AvgBathroom = DF.groupby('propertyType').agg(F.avg('bathrooms').alias('AvgBathRoom'))

#AvgBathroom.show(10)

MaxAvg = AvgBathroom.orderBy('AvgBathRoom', ascending=False).limit(1)

MaxAvg.show()

+-----------------+-----------+
|     propertyType|AvgBathRoom|
+-----------------+-----------+
|Detached Property|    3.15625|
+-----------------+-----------+



In [11]:
#3. Calculate the total number of properties available in each country.

DF.select('country').distinct().show()

TotalNoPrroperty = DF.groupby('country').agg(F.count('propertyType').alias('TotalNoPrroperty'))
TotalNoPrroperty.show()

+-------+
|country|
+-------+
|England|
+-------+

+-------+----------------+
|country|TotalNoPrroperty|
+-------+----------------+
|England|          282197|
+-------+----------------+



In [12]:
#4. Find the average percentage change in sale price (saleEstimate_valueChange.percentageChange) for each tenure type.

DF.select(F.col('`saleEstimate_valueChange.percentageChange`'),F.col('tenure') )

AvgForTenur = DF.groupby('tenure').agg(F.avg(F.col('`saleEstimate_valueChange.percentageChange`')).alias('AvgForEachTenur'))

AvgForTenur.show()

+---------+------------------+
|   tenure|   AvgForEachTenur|
+---------+------------------+
|     null| 73.46561198747484|
|   Shared|28.081120839244473|
| Freehold|27.743534334165474|
|   Feudal| 41.31818881187084|
|Leasehold|26.213921361408914|
+---------+------------------+



In [13]:
#5. Identify the country with the highest average rent price (rentEstimate_currentPrice). 


AvgRent = DF.groupby('country').agg(F.avg('rentEstimate_currentPrice').alias('AvgRent'))

MaxAvgRent = AvgRent.orderBy('AvgRent', ascending=False).limit(1)

MaxAvgRent.show()




+-------+-----------------+
|country|          AvgRent|
+-------+-----------------+
|England|3589.369252618498|
+-------+-----------------+



In [14]:
#6. Find the property type (propertyType) with the highest average number of bedrooms. 

#DF.select('propertyType', 'bedrooms').show(10)

AvgBedrooms = DF.groupby('propertyType').agg(F.avg('bedrooms').alias('AvgBedrooms'))

MaxAvg = AvgBedrooms.orderBy('AvgBedrooms', ascending=False).limit(1)

MaxAvg.show()





+--------------+----------------+
|  propertyType|     AvgBedrooms|
+--------------+----------------+
|Detached House|4.12483745123537|
+--------------+----------------+



In [15]:
#7. Calculate the median sale price (saleEstimate_currentPrice) for each tenure type. 

MedianSP = DF.groupby('tenure').agg(F.percentile_approx('saleEstimate_currentPrice', 0.5).alias('MedianSP'))

MedianSP.show()

+---------+--------+
|   tenure|MedianSP|
+---------+--------+
|     null|635000.0|
|   Shared|666000.0|
| Freehold|869000.0|
|   Feudal|553000.0|
|Leasehold|512000.0|
+---------+--------+

